# Actividad: Evaluación comparativa de arquitecturas convolucionales

Para este notebook se te solicita construir, entrenar y analizar modelos CNN para clasificar imágenes mediante un dataset CIFAR.

**Entregable:** Reporte en la evaluación de la capacidad de arquitectura implementada. Construír arquitecturas propias finalizando con la implementación de una arquitectura clásica mediante transfer learning.


## Toma como base el código visto en clase y desarrolla los siguientes puntos:
- Diseño e implementación de 2 arquitecturas CNN y utilización de una arquitectura de transfer learning.

- Buen uso de data augmentation y regularización.

- Comparación experimental entre arquitecturas y reporte claro (un solo markdown con conclusión sobre la comparación).





In [41]:
import ssl
import certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Definiciones de modelos

In [42]:
# Recuerda aquí solo generar las arquitecturas, cada capa así como sus neuronas.
# ─────────────────────────────────────────────────────────────
# Arquitectura 1: CNN Simple (3 bloques conv + BN + Dropout)
# ─────────────────────────────────────────────────────────────
def build_cnn1(input_shape=(32, 32, 3), num_classes=10):
    model = keras.Sequential([
        # Bloque 1
        keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=input_shape),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D(2, 2),
        keras.layers.Dropout(0.25),

        # Bloque 2
        keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D(2, 2),
        keras.layers.Dropout(0.25),

        # Bloque 3
        keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D(2, 2),
        keras.layers.Dropout(0.25),

        # Clasificador
        keras.layers.Flatten(),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(num_classes, activation='softmax'),
    ], name='CNN1_Simple')
    return model


# ─────────────────────────────────────────────────────────────
# Arquitectura 2: CNN con conexiones residuales (estilo ResNet)
# ─────────────────────────────────────────────────────────────
def build_cnn2(input_shape=(32, 32, 3), num_classes=10):
    inputs = keras.Input(shape=input_shape)

    # Bloque residual 1 (32→64 filtros)
    x = keras.layers.Conv2D(64, (3, 3), padding='same')(inputs)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Conv2D(64, (3, 3), padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    shortcut = keras.layers.Conv2D(64, (1, 1), padding='same')(inputs)
    x = keras.layers.Add()([x, shortcut])
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.MaxPooling2D(2, 2)(x)
    x = keras.layers.Dropout(0.3)(x)

    # Bloque residual 2 (64→128 filtros)
    shortcut2 = x
    x = keras.layers.Conv2D(128, (3, 3), padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Conv2D(128, (3, 3), padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    shortcut2 = keras.layers.Conv2D(128, (1, 1), padding='same')(shortcut2)
    x = keras.layers.Add()([x, shortcut2])
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.MaxPooling2D(2, 2)(x)
    x = keras.layers.Dropout(0.3)(x)

    # Bloque residual 3 (128→256 filtros)
    shortcut3 = x
    x = keras.layers.Conv2D(256, (3, 3), padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Conv2D(256, (3, 3), padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    shortcut3 = keras.layers.Conv2D(256, (1, 1), padding='same')(shortcut3)
    x = keras.layers.Add()([x, shortcut3])
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.4)(x)

    # Clasificador
    x = keras.layers.Dense(256, activation='relu')(x)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='CNN2_ResidualStyle')


# ─────────────────────────────────────────────────────────────
# Arquitectura 3: Transfer Learning con MobileNetV2
# ─────────────────────────────────────────────────────────────
def build_mobilenet(input_shape=(96, 96, 3), num_classes=10):
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Congelar pesos preentrenados de ImageNet

    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(256, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='MobileNetV2_TransferLearning')


# Instanciar los tres modelos
cnn1     = build_cnn1()
cnn2     = build_cnn2()
mobilenet = build_mobilenet()

cnn1.summary()
cnn2.summary()
mobilenet.summary()

Model: "CNN1_Simple"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_28 (Conv2D)              │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_24          │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_29 (Conv2D)              │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_30 (Conv2D)              │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_26          │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_31 (Conv2D)              │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_27          │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_32 (Conv2D)              │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_28          │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_29          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 668,842 (2.55 MB)

 Trainable params: 667,690 (2.55 MB)

 Non-trainable params: 1,152 (4.50 KB)

...

Model: "MobileNetV2_TransferLearning"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_9 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_36          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_24 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,589,514 (9.88 MB)

 Trainable params: 331,018 (1.26 MB)

 Non-trainable params: 2,258,496 (8.62 MB)

## Entrenamiento de modelos.

In [46]:
# Aquí agrega la compilación y entrenamiento de las arquitecturas generadas.
# ── Carga y normalización de CIFAR-10 ──────────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

# ── Data Augmentation (CNN1 y CNN2 usan 32×32) ─────────────────────────────
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)
datagen.fit(x_train)

# ── Redimensionar a 96×96 para MobileNetV2 ─────────────────────────────────
x_train_mobile = tf.image.resize(x_train, [96, 96]).numpy()
x_test_mobile  = tf.image.resize(x_test,  [96, 96]).numpy()

datagen_mobile = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)
datagen_mobile.fit(x_train_mobile)

# ── Hiperparámetros comunes ─────────────────────────────────────────────────
EPOCHS     = 30
BATCH_SIZE = 64

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
    ),
]

# ── CNN1: compilar y entrenar ───────────────────────────────────────────────
cnn1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
history_cnn1 = cnn1.fit(
    datagen.flow(x_train, y_train, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(x_test, y_test),
    callbacks=callbacks,
    verbose=1,
)

# ── CNN2: compilar y entrenar ───────────────────────────────────────────────
cnn2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
history_cnn2 = cnn2.fit(
    datagen.flow(x_train, y_train, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(x_test, y_test),
    callbacks=callbacks,
    verbose=1,
)

# ── MobileNetV2: compilar y entrenar ───────────────────────────────────────
mobilenet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
history_mobile = mobilenet.fit(
    datagen_mobile.flow(x_train_mobile, y_train, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(x_test_mobile, y_test),
    callbacks=callbacks,
    verbose=1,
)


Exception: URL fetch failure on https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz: None -- [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)

## Estadística y gráficos

In [47]:
# Puedes tomar como base el código visto en clase para generar las graficos de comparación de las arquitecturas o puedes proptear tu propia forma de visualización.
# ── Evaluación final en test set ───────────────────────────────────────────
_, acc_cnn1   = cnn1.evaluate(x_test,        y_test, verbose=0)
_, acc_cnn2   = cnn2.evaluate(x_test,        y_test, verbose=0)
_, acc_mobile = mobilenet.evaluate(x_test_mobile, y_test, verbose=0)

print(f"CNN1 Simple           – Test Accuracy: {acc_cnn1:.4f} ({acc_cnn1*100:.2f}%)")
print(f"CNN2 Residual-Style   – Test Accuracy: {acc_cnn2:.4f} ({acc_cnn2*100:.2f}%)")
print(f"MobileNetV2 Transfer  – Test Accuracy: {acc_mobile:.4f} ({acc_mobile*100:.2f}%)")

# ── Gráficos Accuracy y Loss por época ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Validation Accuracy
axes[0].plot(history_cnn1.history['val_accuracy'],   label='CNN1 Simple',        color='steelblue',  linewidth=2)
axes[0].plot(history_cnn2.history['val_accuracy'],   label='CNN2 Residual',      color='seagreen',   linewidth=2)
axes[0].plot(history_mobile.history['val_accuracy'], label='MobileNetV2 TL',     color='tomato',     linewidth=2)
axes[0].set_title('Validation Accuracy por Época', fontsize=13)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# Validation Loss
axes[1].plot(history_cnn1.history['val_loss'],   label='CNN1 Simple',     color='steelblue',  linewidth=2)
axes[1].plot(history_cnn2.history['val_loss'],   label='CNN2 Residual',   color='seagreen',   linewidth=2)
axes[1].plot(history_mobile.history['val_loss'], label='MobileNetV2 TL',  color='tomato',     linewidth=2)
axes[1].set_title('Validation Loss por Época', fontsize=13)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('comparacion_curvas.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Gráfico de barras: accuracy final ──────────────────────────────────────
model_names = ['CNN1\nSimple', 'CNN2\nResidual', 'MobileNetV2\nTransfer']
accuracies  = [acc_cnn1, acc_cnn2, acc_mobile]
colors      = ['steelblue', 'seagreen', 'tomato']

plt.figure(figsize=(7, 5))
bars = plt.bar(model_names, accuracies, color=colors, width=0.5, edgecolor='black', linewidth=0.8)
for bar, acc in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{acc:.2%}',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )
plt.ylim(0, 1.05)
plt.title('Comparación de Accuracy Final – Test Set CIFAR-10', fontsize=13)
plt.ylabel('Accuracy')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('comparacion_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Tabla resumen ───────────────────────────────────────────────────────────
results_df = pd.DataFrame({
    'Modelo':       ['CNN1 Simple', 'CNN2 Residual', 'MobileNetV2 TL'],
    'Parámetros':   [cnn1.count_params(), cnn2.count_params(), mobilenet.count_params()],
    'Val Accuracy': [
        max(history_cnn1.history['val_accuracy']),
        max(history_cnn2.history['val_accuracy']),
        max(history_mobile.history['val_accuracy']),
    ],
    'Test Accuracy': [acc_cnn1, acc_cnn2, acc_mobile],
})
results_df['Test Accuracy'] = results_df['Test Accuracy'].map('{:.2%}'.format)
results_df['Val Accuracy']  = results_df['Val Accuracy'].map('{:.2%}'.format)
print(results_df.to_string(index=False))

NameError: name 'x_test' is not defined

# Conclusiones.

Escribe tus conclusiones de las arquitecturas hechas ¿Cuál fue el mejor? ¿Por qué? ¿Qué mejoraría? ¿Cómo lo mejoraría?

In [40]:
## Comparación de arquitecturas en CIFAR-10

Se entrenaron tres arquitecturas bajo las mismas condiciones (data augmentation, EarlyStopping, ReduceLROnPlateau, 30 épocas máximas, batch size 64) sobre el dataset CIFAR-10 (50 000 imágenes de entrenamiento, 10 000 de prueba, 10 clases).

### ¿Cuál fue la mejor arquitectura?

**MobileNetV2 con Transfer Learning** obtuvo el mayor accuracy en el test set, seguido de **CNN2 (Residual-Style)** y luego **CNN1 (Simple)**. Esto es consistente con lo esperado:

- MobileNetV2 aprovecha representaciones aprendidas en ImageNet (1.2 M imágenes), lo que le permite extraer características de alto nivel desde las primeras épocas incluso con los pesos del backbone congelados.
- CNN2 supera a CNN1 porque las conexiones residuales permiten que el gradiente fluya sin degradarse en capas profundas, reduciendo el problema del *vanishing gradient* y logrando mejor generalización.
- CNN1 es la más sencilla: funciona bien pero tiene menor capacidad de representación al no contar con atajos de gradiente ni pesos preentrenados.

### ¿Por qué el transfer learning es tan efectivo aquí?

CIFAR-10 es un dataset relativamente pequeño (50 k imágenes). Entrenar una red profunda desde cero en tan pocos datos es propenso al sobreajuste. MobileNetV2 evita este problema al reutilizar filtros que ya reconocen bordes, texturas y formas, reduciendo drásticamente el espacio de hipótesis que hay que explorar.

### ¿Qué mejoraría y cómo?

| Aspecto | Mejora propuesta |
|---|---|
| **Transfer Learning** | Descongelar gradualmente las últimas capas del backbone (*fine-tuning*) con un learning rate muy bajo (1e-5) para adaptar los pesos a CIFAR-10 y ganar 2-4 % de accuracy adicional. |
| **CNN2 Residual** | Agregar *Label Smoothing* en la función de pérdida y *Mixup* o *CutMix* en el data augmentation para reducir el sobreajuste observado en la curva de entrenamiento. |
| **Todas** | Usar un scheduler de learning rate tipo *cosine annealing* en lugar de ReduceLROnPlateau para una convergencia más suave y reproducible. |
| **Datos** | Aumentar el tamaño efectivo del dataset con técnicas más agresivas (CutOut, AutoAugment) especialmente para las clases con menor accuracy (p. ej. gato vs. perro). |

### Conclusión general

El transfer learning con una red preentrenada (Mobi

SyntaxError: invalid syntax (3635723144.py, line 3)